# Advanced Predictive Models: Lasso, Trees, Random Forests, and Boosting

---
This notebook shows how to use more advanced machine learning models in Python for predicting customer behaviour in banking. We use the same bank marketing data as in Units 01 and 02: the outcome `y` is 1 if the customer subscribed to a term deposit after the call. We cover:

- **Lasso regression**: A linear model that automatically selects important features
- **Decision Trees**: Easy-to-interpret models that make decisions like a flowchart
- **Random Forests**: Combines many decision trees for better predictions
- **Boosting**: Builds models sequentially, learning from previous mistakes

## Learning Objectives
By the end of this notebook, you will be able to:
1. Choose a feature set with the decision-time rule and explain why `duration` is dropped.
2. Fit a cross-validated Lasso, a shallow tree, a random forest and a boosted model on one train/test split.
3. Show, with numbers, why a single deep tree overfits and why averaging many trees helps.
4. Compare models with accuracy, AUC, recall and precision against the majority-class baseline, and read the results as a call-centre manager would.
5. Explain the difference between "important for prediction" and "causes subscription", and state what a causal estimate would need to be believed.

All packages used in this notebook are pre-installed on Google Colab. Section 6 (causal analysis) is optional and takes about five minutes to run.


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,
                             precision_score, confusion_matrix, roc_curve)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Section 6 (causal analysis) is optional and takes a few minutes.
# Set RUN_CAUSAL = False to skip its code cells.
RUN_CAUSAL = True

print("Setup complete.")


In [ ]:
# Load and explore the banking dataset
data = pd.read_csv("https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv")

print(f"Dataset dimensions: {len(data)} rows and {data.shape[1]} columns\n")
data.info()

print("\nMissing values per column:")
print(data.isnull().sum())

# Look at the target variable distribution
print("\nTarget variable (y) distribution:")
print(data['y'].value_counts())
print(f"\nShare of subscribers: {data['y'].mean():.3f}")


## Data Preparation

Before building models we choose the feature set and make one train/test split that every model in this notebook will use.

**Which columns are known at decision time?** The question a model in this unit answers is: "before we call this customer, how likely is it that they subscribe?" A feature is allowed only if the bank knows it at that moment.

- `duration` is the length of the call. It is known only after the call ends, and a call of length zero cannot end in a subscription. Unit 02 dropped it for the same reason (leakage). We drop it here too. Section 3 has an optional demonstration of what happens if you keep it.
- `campaign` (contacts so far in this campaign), `pdays`, `previous` and `poutcome` (history of earlier campaigns) are in the customer file before the call, so they stay.
- The five macro indicators (`emp_var_rate` to `nr_employed`) are published figures and stay.

We also convert the text columns to categorical type and encode them as 0/1 dummy columns, because scikit-learn models need numeric input. The same helper function scores every model on the same test set, so the comparison in Section 5 is fair.


In [ ]:
# Feature set: everything the bank knows before the call
data_full = data.copy()                 # kept only for the optional demonstrations later on
data = data.drop(columns=['duration'])

categorical_vars = ["marital", "education", "housing", "loan", "contact", "poutcome"]
for col in categorical_vars:
    data[col] = data[col].astype('category')

# One train/test split (80/20), stratified so both sets have the same share of subscribers
train_data, test_data = train_test_split(data, test_size=0.2, random_state=123, stratify=data['y'])
print(f"Training set: {len(train_data)} observations")
print(f"Test set: {len(test_data)} observations")

# Dummy-encode the categorical columns; align the test columns to the training columns
X_train = pd.get_dummies(train_data.drop('y', axis=1), drop_first=True, dtype=float)
X_test = pd.get_dummies(test_data.drop('y', axis=1), drop_first=True, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train_data['y'].astype(int)
y_test = test_data['y'].astype(int)
print(f"\nNumber of features after dummy encoding: {X_train.shape[1]}")

# The baseline every model must beat: always predict "no"
majority_rate = (y_test == 0).mean()
print(f"\nShare of subscribers in the test set: {y_test.mean():.3f}")
print(f"Majority-class baseline accuracy (predict 'no' for everyone): {majority_rate:.3f}")

# One evaluation helper used for every model, so all models are scored the same way
results = {}   # model name -> dict of metrics, filled in as we go

def evaluate(name, prob, threshold=0.5):
    """Score predicted probabilities on the test set, print them next to the baseline, store them."""
    pred = (prob > threshold).astype(int)
    m = {
        'Accuracy': accuracy_score(y_test, pred),
        'AUC': roc_auc_score(y_test, prob),
        'Recall (yes)': recall_score(y_test, pred, zero_division=0),
        'Precision (yes)': precision_score(y_test, pred, zero_division=0),
    }
    results[name] = m
    print(f"\n{name} on the test set (threshold {threshold}):")
    print(f"  Accuracy:  {m['Accuracy']:.3f}   (majority baseline: {majority_rate:.3f})")
    print(f"  AUC:       {m['AUC']:.3f}   (random guessing: 0.500)")
    print(f"  Recall on subscribers:    {m['Recall (yes)']:.3f}   (share of actual subscribers the model flags)")
    print(f"  Precision on subscribers: {m['Precision (yes)']:.3f}   (share of flagged customers who subscribe)")
    cm = confusion_matrix(y_test, pred)
    print("  Confusion matrix:")
    print(pd.DataFrame(cm, columns=['Predicted 0', 'Predicted 1'],
                       index=['Actual 0', 'Actual 1']).to_string())


## 1. Lasso Regression

**What is Lasso?** Lasso (Least Absolute Shrinkage and Selection Operator) is a regression (here a logistic regression) with a penalty on the absolute size of the coefficients. The penalty pushes small coefficients exactly to zero, so the model selects features on its own. That reduces overfitting and makes the model easier to read.

**When to use Lasso:**
- When you have many features and want automatic feature selection
- When you need an interpretable model
- When you suspect many features are irrelevant

**Implementation details:**
- Features are standardised first (`StandardScaler`). The penalty treats every coefficient the same, so every feature must be on the same scale.
- `LogisticRegressionCV` tries 20 values of the penalty strength `C` (a small `C` means a strong penalty) and keeps the one with the best cross-validated AUC. This mirrors `cv.gamlr` in R, where the penalty is called lambda and a large lambda means a strong penalty.
- The `saga` solver supports the L1 penalty on a data set of this size.

### Logistic Lasso: Predicting customer subscription (y)


In [ ]:
# Standardise the features (fit the scaler on the training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cross-validated Lasso: 20 values of C, 5 folds, keep the C with the best AUC.
# scikit-learn renamed the way the L1 penalty is requested in version 1.8 (penalty='l1' became
# l1_ratios=[1.0]) and simplified the fitted attributes in 1.10, so we pick the arguments by version.
import sklearn
skl_version = tuple(int(v) for v in sklearn.__version__.split('.')[:2])
lasso_kwargs = dict(solver='saga', Cs=20, cv=5, scoring='roc_auc', max_iter=2000, random_state=123, n_jobs=-1)
if skl_version >= (1, 8):
    lasso_kwargs['l1_ratios'] = [1.0]            # pure L1 penalty
    if skl_version == (1, 9):
        lasso_kwargs['use_legacy_attributes'] = False   # opt in to the 1.10 attribute layout (silences a warning)
else:
    lasso_kwargs['penalty'] = 'l1'

print("Training Lasso (L1-penalised logistic regression) with 5-fold cross-validation...")
lasso_model = LogisticRegressionCV(**lasso_kwargs)
lasso_model.fit(X_train_scaled, y_train)
chosen_C = float(np.ravel(lasso_model.C_)[0])
print(f"Chosen C (best cross-validated AUC): {chosen_C:.4f}")

# The cross-validation results per fold and per C. The attribute layout depends on the scikit-learn
# version: a dict keyed by class (legacy, sometimes with an extra l1_ratios axis) or a plain array.
Cs = lasso_model.Cs_
scores = lasso_model.scores_
if isinstance(scores, dict):                          # legacy layout
    scores = scores[lasso_model.classes_[1]]
    if scores.ndim == 3:                              # (n_folds, n_Cs, n_l1_ratios): drop the l1_ratios axis
        scores = scores[:, :, 0]
else:                                                 # new layout: (n_folds, n_l1_ratios, n_Cs)
    scores = scores[:, 0, :]
paths = lasso_model.coefs_paths_
if isinstance(paths, dict):                           # legacy layout
    paths = paths[lasso_model.classes_[1]]
    if paths.ndim == 4:                               # (n_folds, n_Cs, n_l1_ratios, p + 1): drop the l1_ratios axis
        paths = paths[:, :, 0, :]
else:                                                 # new layout: (n_folds, n_l1_ratios, n_Cs, n_classes, p + 1)
    paths = paths[:, 0, :, 0, :]
paths = paths[:, :, :-1]                              # drop the intercept; now (n_folds, n_Cs, p)
n_nonzero = (np.abs(paths) > 1e-8).sum(axis=2).mean(axis=0)   # mean over folds
cv_auc = scores.mean(axis=0)
cv_se = scores.std(axis=0, ddof=1) / np.sqrt(scores.shape[0])

path = pd.DataFrame({'C': Cs,
                     'Non-zero coefficients (mean over folds)': np.round(n_nonzero, 1),
                     'CV AUC': np.round(cv_auc, 4)})
print("\nRegularisation path (strong penalty at the top, weak penalty at the bottom):")
print(path.to_string(index=False))

# The one-standard-error rule (what gamlr calls lambda.1se): the strongest penalty whose CV AUC
# is within one standard error of the best. It prefers the simpler model when the curve is flat.
best = int(np.argmax(cv_auc))
one_se_idx = int(np.min(np.where(cv_auc >= cv_auc[best] - cv_se[best])[0]))
print(f"\nOne-standard-error choice: C = {Cs[one_se_idx]:.4f} with about {n_nonzero[one_se_idx]:.0f} non-zero coefficients "
      f"(CV AUC {cv_auc[one_se_idx]:.4f} versus {cv_auc[best]:.4f} at the best C)")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(Cs, n_nonzero, marker='o', color='steelblue')
ax1.set_xscale('log')
ax1.set_xlabel('C (inverse penalty strength, log scale)')
ax1.set_ylabel('Non-zero coefficients', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(Cs, cv_auc, marker='s', color='darkorange')
ax2.set_ylabel('Cross-validated AUC', color='darkorange')
ax1.axvline(chosen_C, color='grey', linestyle='--', label='chosen C')
ax1.axvline(Cs[one_se_idx], color='grey', linestyle=':', label='one-SE choice')
ax1.legend(loc='lower right')
plt.title('Lasso path: how many features survive as the penalty weakens')
plt.tight_layout()
plt.show()

# Coefficients of the chosen model
lasso_coef = pd.Series(lasso_model.coef_[0], index=X_train.columns)
selected_features = lasso_coef[lasso_coef != 0]
print(f"\nNumber of selected features at the chosen C: {len(selected_features)} out of {len(lasso_coef)}")
print("\nLargest coefficients (standardised scale):")
print(lasso_coef.reindex(lasso_coef.abs().sort_values(ascending=False).index).head(10).round(3))

# Predictions on the test set
lasso_pred_prob = lasso_model.predict_proba(X_test_scaled)[:, 1]
evaluate('Lasso', lasso_pred_prob)


## 2. Decision Trees

**What are Decision Trees?** Decision trees make predictions by asking a series of yes/no questions about the features. They are like a flowchart that leads to a prediction.

**When to use Decision Trees:**
- When you need a highly interpretable model
- When relationships between features are non-linear
- When you want to understand the decision-making process

**Pros:** Easy to read, no scaling needed, captures non-linear patterns and interactions.
**Cons:** A tree grown without limits memorises the training data (overfits), and small changes in the data can produce a very different tree.

We first fit a small tree with at most three levels and at least 100 customers per leaf, so that we can read it. R's `rpart` uses a different stopping rule by default (a relative cost-complexity parameter `cp`); we set the same depth and leaf-size limits in both languages instead, so the trees are comparable.


In [ ]:
# A shallow, readable tree
tree_mod = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=123)
tree_mod.fit(X_train, y_train)

plt.figure(figsize=(20, 9))
plot_tree(tree_mod, feature_names=X_train.columns, class_names=['no', 'yes'],
          filled=True, rounded=True, fontsize=9, proportion=True)
plt.title("Decision Tree for Customer Subscription Prediction (depth 3)")
plt.show()

tree_pred_prob = tree_mod.predict_proba(X_test)[:, 1]
evaluate('Decision Tree', tree_pred_prob)

# Feature importance: share of the impurity reduction attributed to each feature.
# It tells you which features the tree used to sort customers, not how much they change the probability.
importance_tree = pd.Series(tree_mod.feature_importances_, index=X_train.columns)
print("\nFeature Importance (Decision Tree):")
print(importance_tree[importance_tree > 0].sort_values(ascending=False).round(3))


### Why we limit the tree: overfitting

What happens if we let the tree grow until every leaf is pure? Compare the AUC on the training data with the AUC on the test data, for the shallow tree and for an unlimited tree.


In [ ]:
# An unlimited tree: no depth limit, a leaf may hold a single customer
deep_tree = DecisionTreeClassifier(random_state=123)
deep_tree.fit(X_train, y_train)

rows = []
for name, model in [('Shallow tree (depth 3, leaf >= 100)', tree_mod), ('Unlimited tree', deep_tree)]:
    rows.append({
        'Model': name,
        'Leaves': model.get_n_leaves(),
        'Depth': model.get_depth(),
        'Train AUC': roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]),
        'Test AUC': roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
    })
overfit_table = pd.DataFrame(rows).round(3)
print(overfit_table.to_string(index=False))


**What the table shows.** In this run the unlimited tree has 5,645 leaves and a training AUC of 0.999, but a test AUC of 0.62, well below the eight-leaf tree's 0.76. It has memorised the training customers, noise included, and that memory does not transfer to new customers. The gap between training and test performance is what overfitting means. Limiting depth and leaf size is one cure. Section 3 shows another: keep the deep trees, but average many of them.


## 3. Random Forests

**What are Random Forests?** A random forest grows many deep trees, each on a bootstrap sample of the training rows, and at each split lets a tree choose among a random subset of the features only. The forest's prediction is the average of the trees' predictions.

**Why does averaging help?** Section 2 showed that a single deep tree has low bias but high variance: it fits the training data almost perfectly and generalises poorly. Averaging many such trees keeps the low bias and cuts the variance, in the same way that the average of many noisy measurements is less noisy than one measurement. Bootstrap samples and random feature subsets make the trees different from each other, and averaging only reduces variance when the trees do not all make the same mistakes.

**When to use Random Forests:**
- When you want better accuracy than a single decision tree with little tuning
- When you have enough data (thousands of rows)
- When you want a feature importance ranking

**Pros:** Usually more accurate than a single tree, few parameters to tune, and the out-of-bag (OOB) score gives an honest performance estimate without a test set.
**Cons:** Not readable as a flowchart, slower to train and predict than a single tree, and in scikit-learn missing values must be handled beforehand (this data set has none).


In [ ]:
# Fit a random forest for classification
mtry = int(np.sqrt(X_train.shape[1]))  # features tried at each split; the usual default for classification
rf_mod = RandomForestClassifier(n_estimators=500, max_features=mtry, oob_score=True,
                                random_state=123, n_jobs=-1)
rf_mod.fit(X_train, y_train)

print(f"Random Forest with {rf_mod.n_estimators} trees, {mtry} features tried per split")
print(f"Out-of-bag accuracy: {rf_mod.oob_score_:.3f}   (majority baseline in training data: {(y_train == 0).mean():.3f})")
print("(each tree is scored on the training rows it did not see, so this needs no test set)")

rf_pred_prob = rf_mod.predict_proba(X_test)[:, 1]
evaluate('Random Forest', rf_pred_prob)

# Feature importance (mean decrease in impurity, averaged over all trees)
importance_rf = pd.Series(rf_mod.feature_importances_, index=X_train.columns)
print("\nTop 10 Most Important Features (Random Forest):")
print(importance_rf.sort_values(ascending=False).head(10).round(3))

plt.figure(figsize=(10, 6))
importance_rf.sort_values(ascending=False).head(15).plot(kind='barh')
plt.xlabel('Feature importance (mean decrease in impurity)')
plt.title('Random Forest Feature Importance (Top 15)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


**Reading the importance ranking.** Feature importance measures how much a feature helped the forest sort customers into subscribers and non-subscribers. It is not an effect size: a high-ranked feature is not "the reason" a customer subscribes, and the ranking says nothing about what would happen if the bank changed that feature. "Important for prediction" and "causes subscription" are different claims. Section 6 returns to this.


### Why many trees: variance reduction

How does the test AUC change as trees are added? We refit the forest with 1, 5, 25, 100 and 500 trees. A forest with one tree is just one deep tree from Section 2, grown on a bootstrap sample.


In [ ]:
n_trees_grid = [1, 5, 25, 100, 500]
auc_by_trees = []
for n in n_trees_grid:
    rf_n = RandomForestClassifier(n_estimators=n, max_features=mtry, random_state=123, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    auc_by_trees.append(roc_auc_score(y_test, rf_n.predict_proba(X_test)[:, 1]))
    print(f"{n:>4} trees: test AUC = {auc_by_trees[-1]:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(n_trees_grid, auc_by_trees, marker='o')
plt.xscale('log')
plt.xlabel('Number of trees (log scale)')
plt.ylabel('Test AUC')
plt.title('Random forest: test AUC as trees are added')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**What the curve shows.** In this run the test AUC rises from 0.62 with one tree to 0.76 with 25 trees and 0.78 with 500; almost all of the gain comes in the first 25 trees, after which more trees only smooth the estimate. Three things make this work. *Bagging:* each tree sees a different bootstrap sample, so the trees' errors are partly independent and averaging cancels them. *Feature subsampling:* at each split a tree may only choose among `mtry` randomly picked features, which stops every tree from opening with the same strongest feature, makes the trees less alike, and so lets averaging remove more variance. *Averaging itself:* each single tree is as overfitted as the unlimited tree of Section 2, yet the average of 500 of them is not, because the noise each tree fitted is different.


### Optional: what if we had kept `duration`?

The data preparation dropped `duration` because the bank does not know the call length before the call. Here is what the forest would report if we had kept it.


In [ ]:
# Refit the forest with duration included (same split, same rows)
train_full = data_full.loc[train_data.index]
test_full = data_full.loc[test_data.index]
X_train_dur = pd.get_dummies(train_full.drop('y', axis=1), drop_first=True, dtype=float)
X_test_dur = pd.get_dummies(test_full.drop('y', axis=1), drop_first=True, dtype=float)
X_test_dur = X_test_dur.reindex(columns=X_train_dur.columns, fill_value=0)

rf_dur = RandomForestClassifier(n_estimators=200, max_features=mtry, random_state=123, n_jobs=-1)
rf_dur.fit(X_train_dur, y_train)
auc_with = roc_auc_score(y_test, rf_dur.predict_proba(X_test_dur)[:, 1])
auc_without = results['Random Forest']['AUC']

print(f"Test AUC without duration: {auc_without:.3f}")
print(f"Test AUC with duration:    {auc_with:.3f}")
print(f"Gap:                       {auc_with - auc_without:+.3f}")

imp_dur = pd.Series(rf_dur.feature_importances_, index=X_train_dur.columns).sort_values(ascending=False)
print("\nTop 3 features with duration included:")
print(imp_dur.head(3).round(3))


**Why is the "better" model useless at decision time?** In this run the AUC jumps from 0.78 to 0.94 once `duration` is included, and `duration` becomes the top feature by a wide margin. But the input the model now relies on does not exist when the decision is made: the bank picks whom to call before the call, and the length of that call is a result of the call, not a property of the customer. A model scored with information from the future looks excellent in a backtest and cannot be run in production. Whenever a feature makes a model look too good, ask when that feature becomes known.


## 4. Gradient Boosting

**What is Gradient Boosting?** Boosting also combines many trees, but not by averaging independent trees. It builds small trees one after another, and each new tree is fitted to the errors the current ensemble still makes. The `learning_rate` scales down each tree's contribution, so the model improves in many small steps.

**Early stopping.** Because every round adds a tree that fits the remaining errors, a boosted model keeps improving on the training data and at some point starts to overfit. We therefore hold out 20 percent of the training data as a validation set, track the validation AUC after every round, and stop when it has not improved for 10 rounds. R's `xgb.cv` does the same with cross-validation.

**When to use Gradient Boosting:**
- When you want the best predictive performance on tabular data
- When you can afford to tune the learning rate, the tree depth and the number of rounds
- When accuracy matters more than interpretability

**Pros:** Often the most accurate model on tabular data. `HistGradientBoostingClassifier` is fast, built into scikit-learn, and handles missing values natively.
**Cons:** More parameters to tune, sensitive to the learning rate, not readable, and it has no built-in impurity importance (we compute a permutation importance instead).


In [ ]:
# Gradient boosting with early stopping on a 20% validation split of the training data
gb_mod = HistGradientBoostingClassifier(
    early_stopping=True, validation_fraction=0.2, n_iter_no_change=10, scoring='roc_auc',
    max_iter=500, learning_rate=0.1, max_depth=4, random_state=123
)
gb_mod.fit(X_train, y_train)
print(f"Rounds allowed: 500; rounds used before early stopping: {gb_mod.n_iter_}")

# Training vs validation AUC by round: what early stopping is looking at
plt.figure(figsize=(8, 5))
plt.plot(gb_mod.train_score_, label='Training AUC')
plt.plot(gb_mod.validation_score_, label='Validation AUC')
plt.axvline(gb_mod.n_iter_, color='grey', linestyle='--', label='stopping point')
plt.xlabel('Boosting round')
plt.ylabel('AUC')
plt.title('Gradient boosting: training vs validation AUC by round')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gb_pred_prob = gb_mod.predict_proba(X_test)[:, 1]
evaluate('Gradient Boosting', gb_pred_prob)

# Permutation importance: how much the test AUC drops when one feature's values are shuffled
perm = permutation_importance(gb_mod, X_test, y_test, scoring='roc_auc',
                              n_repeats=5, random_state=123, n_jobs=-1)
importance_gb = pd.Series(perm.importances_mean, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 features by permutation importance (drop in test AUC when shuffled):")
print(importance_gb.head(10).round(4))

plt.figure(figsize=(10, 6))
importance_gb.head(15).plot(kind='barh')
plt.xlabel('Drop in test AUC when the feature is shuffled')
plt.title('Gradient Boosting Permutation Importance (Top 15)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, gb_pred_prob)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f"Gradient boosting (AUC = {results['Gradient Boosting']['AUC']:.3f})")
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random guessing (AUC = 0.500)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate (recall)')
plt.title('ROC Curve (Gradient Boosting)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 5. Model Comparison

All four models are scored on the same test set at the same threshold of 0.5. The first row is the baseline that predicts "no" for everyone.

- **Accuracy** is the share of correct predictions. With 11 percent subscribers, "always no" already gets 89 percent, so accuracy alone cannot tell the models apart.
- **AUC** is the probability that a randomly chosen subscriber gets a higher predicted probability than a randomly chosen non-subscriber. It does not depend on the threshold and is the main number to compare here.
- **Recall on subscribers** is the share of actual subscribers the model flags. For a call centre: of all customers who would say yes, how many does the model put on the call list?
- **Precision on subscribers** is the share of flagged customers who subscribe. For a call centre: of the customers on the list, how many calls end in a sale?


In [ ]:
# Comparison table: baseline row plus all four models, same test set, threshold 0.5
baseline = pd.DataFrame({'Accuracy': [majority_rate], 'AUC': [0.5],
                         'Recall (yes)': [0.0], 'Precision (yes)': [np.nan]},
                        index=['Majority baseline (always no)'])
model_comparison = pd.concat([baseline, pd.DataFrame(results).T]).round(3)
model_comparison.index.name = 'Model'

print("Model comparison on the same test set (threshold 0.5; precision undefined for the baseline):")
print(model_comparison.to_string())

# Plot the AUC of the four models
plot_df = (model_comparison.drop(index='Majority baseline (always no)')
           .reset_index().sort_values('AUC', ascending=False).reset_index(drop=True))
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, y='Model', x='AUC', hue='Model', palette='viridis', legend=False)
plt.axvline(0.5, color='grey', linestyle='--', label='random guessing')
for i, row in plot_df.iterrows():          # i is the bar position because the index was reset
    plt.text(row['AUC'] + 0.005, i, f"{row['AUC']:.3f}", va='center')
plt.xlim(0.4, 1.0)
plt.xlabel('Test AUC')
plt.title('Model comparison by test AUC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Reading the table as a call-centre manager

In this run:

- **Accuracy** puts every model between 0.89 and 0.90, at most 1.6 points above the baseline of 0.887. Accuracy cannot separate the models on data with 11 percent subscribers, so "90 percent accuracy" in a report tells the manager almost nothing.
- **AUC** separates them: gradient boosting 0.80, random forest 0.78, Lasso 0.78, the shallow tree 0.76, against 0.50 for guessing. The boosted model ranks customers best, and the gap between the single tree and the ensembles is what Sections 2 to 4 led us to expect.
- **Recall on subscribers** at the 0.5 threshold is low for every model (0.19 to 0.31). At this threshold the call list holds only the customers the model is very sure about, so most customers who would say yes are never called. The random forest calls the most of them.
- **Precision on subscribers** is about 0.70 for the Lasso, the tree and the boosted model: seven of ten customers on their short lists subscribe, against 11 in 100 for a random call. The forest's longer list buys more recall at the cost of precision (0.55).

The threshold of 0.5 is a convention, not a business decision. A call centre with the budget to call 20 percent of the customers should lower the threshold until the list has the right length and then compare the models' precision at that list length. Exercise 1 does this.

### Model characteristics, as implemented here

1. **Lasso** (`LogisticRegressionCV`, L1 penalty, standardised features)
   - Selects features by setting coefficients to zero; the path in Section 1 shows the count growing as the penalty weakens
   - Coefficients are readable as directions and relative sizes (on the standardised scale)
   - Fast; the cross-validation over 20 penalties is the slowest part
   - Linear in the features: thresholds and interactions must be built by hand
   - With 26 features and 33,000 rows there is little to select: in this run the best-AUC choice keeps all 26, while the one-standard-error choice keeps 17 at almost the same AUC (0.779 versus 0.780)

2. **Decision Tree** (`DecisionTreeClassifier`, depth 3, at least 100 customers per leaf)
   - Readable as a flowchart; no scaling needed
   - Finds thresholds and interactions on its own
   - Overfits badly without limits (Section 2: training AUC 0.999 versus test AUC 0.62)
   - Unstable: small changes in the data can change the top split

3. **Random Forest** (`RandomForestClassifier`, 500 trees, `sqrt(p)` features per split)
   - Averages many deep trees; the variance falls as trees are added (Section 3)
   - Works with the defaults; the out-of-bag score is an honest estimate at no cost
   - Gives an importance ranking, which is not an effect size
   - Not readable; slower; in scikit-learn missing values must be filled beforehand

4. **Gradient Boosting** (`HistGradientBoostingClassifier`, early stopping on a validation split)
   - Best AUC in this run; trees are fitted one after another to the remaining errors
   - Early stopping picks the number of rounds from the data (51 of 500 allowed, in this run)
   - Handles missing values natively; fast on large data
   - More parameters to tune (learning rate, depth, rounds); no built-in impurity importance, so we used permutation importance

### Choosing a model

- Need to explain the rule to a colleague or a regulator? Tree or Lasso.
- Want the best ranking with little tuning? Random forest.
- Want the best ranking and can tune? Gradient boosting with early stopping.
- Whatever you pick, report the AUC and the precision and recall at the list length you will actually use, next to the baseline.


## Exercises

1. **A call budget.** The call centre can afford to call 1,000 of the test customers. Rank the test customers by the random forest's predicted probability and select the top 1,000. Deliverable: report how many of them subscribed, how many subscribers a random selection of 1,000 would find on average (subscriber share times 1,000), and one sentence on what this lift means for the manager.
2. **Tree depth.** Fit trees with `max_depth` from 1 to 10, keeping `min_samples_leaf=100`. Deliverable: a table of training and test AUC by depth and one sentence on the depth at which overfitting starts.
3. **Learning rate.** Refit the boosted model with `learning_rate` 0.01, 0.1 and 0.3, early stopping on. Deliverable: the number of rounds used and the test AUC for each, plus one sentence on how the learning rate and the number of rounds trade off.
4. **Explain accuracy to a manager.** The shallow tree's accuracy is close to the majority baseline. Deliverable: three sentences, for a manager with no statistics background, on why accuracy is the wrong yardstick on this data and which number in the comparison table to look at instead.
5. **Argue before you code.** The bank asks whether calling a customer more than once in a campaign (`campaign > 1`) raises the chance of a subscription. Before writing any code, argue whether repeated calling is as good as random given the pre-treatment covariates. Name one confounder, and one way in which the outcome itself could drive the treatment. Deliverable: one paragraph and a verdict on whether an analysis like the one in Section 6 could answer the question with this data.


---

# 6. From Prediction to Causal Questions (optional, about 5 minutes runtime)

The code cells in this section run only if `RUN_CAUSAL = True` in the setup cell. Set it to `False` to skip them.

## Prediction versus intervention

Every model above answers a prediction question: given what the bank knows before the call, how likely is a subscription? The random forest tells us that age, the macro indicators and the customer's campaign history matter most for that prediction. It does not tell us what would happen if the bank changed anything.

Take the optional demonstration in Section 3: with `duration` included, the forest says call length is by far the most important feature. Can the bank raise subscriptions by making calls longer? No. Long calls happen because the customer is interested, not the other way round. "Important for prediction" is a statement about correlations in the data as it was generated. "Causes subscription" is a statement about what happens after an intervention, and it needs a different kind of argument.

## A treatment the bank actually sets

The bank chooses the channel for each call: mobile phone (`contact == "cellular"`) or landline (`contact == "telephone"`). That is an intervention, so "does calling on a mobile phone raise subscriptions?" is a causal question the bank could act on.

**What we would need to believe.** To read a comparison of the two channels as a causal effect, two assumptions must hold:

1. *Unconfoundedness given pre-treatment covariates.* Among customers with the same pre-treatment characteristics, the channel was chosen as if at random. Nothing that affects both the channel choice and the subscription decision is left out.
2. *Overlap.* For every kind of customer in the comparison, both channels were actually used. Where one group never received a landline call, there is nothing to compare.

Both are assumptions, not facts we can verify from the data. The channel may itself depend on customer characteristics (which customers gave the bank a mobile number, which period the call took place in), so treat the estimates below as an illustration of the method, not as a result to act on.

**Pre-treatment covariates only.** We adjust for `age`, `marital`, `education`, `housing`, `loan` and the five macro indicators, all fixed before the channel was chosen. We deliberately exclude:

- `duration`, `campaign`: realised during the current campaign, after the channel decision. Conditioning on them would adjust away part of the effect.
- `pdays`, `previous`, `poutcome`: results of earlier campaigns, which were themselves run over a channel. They are partly consequences of past channel choices.

## Plan

1. Naive difference in subscription rates between the two channels.
2. Cross-fitted regression adjustment with propensity weighting (AIPW, also called the doubly robust estimator), built from scikit-learn only.
3. An overlap check on the estimated propensity scores.


In [ ]:
if RUN_CAUSAL:
    # Treatment: was the customer called on a mobile phone (1) or a landline (0)?
    causal_data = data_full.copy()
    W = (causal_data['contact'] == 'cellular').astype(int).values
    Y = causal_data['y'].astype(int).values

    # Pre-treatment covariates only: fixed before the channel was chosen
    pre_treatment = ['age', 'marital', 'education', 'housing', 'loan',
                     'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']
    X_causal = pd.get_dummies(causal_data[pre_treatment], drop_first=True, dtype=float)
    print(f"Rows: {len(W)}, covariates after dummy encoding: {X_causal.shape[1]}")
    print(f"Treated (cellular): {W.sum()}, control (telephone): {(1 - W).sum()}")

    # Naive comparison: difference in subscription rates
    p1, p0 = Y[W == 1].mean(), Y[W == 0].mean()
    naive_ate = p1 - p0
    se_naive = np.sqrt(p1 * (1 - p1) / (W == 1).sum() + p0 * (1 - p0) / (W == 0).sum())
    print(f"\nSubscription rate, cellular:  {p1:.3f}")
    print(f"Subscription rate, telephone: {p0:.3f}")
    print(f"Naive difference in means:    {naive_ate:.3f}  (SE {se_naive:.3f})")
    print("This compares different customers in different periods. It is not yet an effect of the channel.")


## Adjusted estimate: cross-fitted AIPW

The adjusted estimator needs three predictions for every customer, all made by models that did **not** see that customer (5-fold cross-fitting, the same idea as cross-validation):

- the propensity score `e(X) = P(cellular | X)`,
- the expected outcome if called on a mobile phone, `mu1(X)`, from a model fitted on treated customers only,
- the expected outcome if called on a landline, `mu0(X)`, from a model fitted on control customers only.

The AIPW score for customer *i* is

`mu1(X_i) - mu0(X_i) + W_i (Y_i - mu1(X_i)) / e(X_i) - (1 - W_i)(Y_i - mu0(X_i)) / (1 - e(X_i))`.

The first part is a regression adjustment; the second corrects it with propensity weights. The average of the scores estimates the average treatment effect, and their standard deviation divided by the square root of the sample size is its standard error. The estimator is "doubly robust": it is consistent if either the outcome models or the propensity model is right. Cross-fitting is what makes the standard error valid when the nuisance models are flexible machine learners.


In [ ]:
if RUN_CAUSAL:
    Xc = X_causal.values
    n = len(Y)
    e_hat = np.zeros(n)     # propensity: P(cellular | X)
    mu1_hat = np.zeros(n)   # E[Y | X, cellular]
    mu0_hat = np.zeros(n)   # E[Y | X, telephone]

    def learner():
        return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=200, random_state=123)

    kf = KFold(n_splits=5, shuffle=True, random_state=123)
    for fold, (fit_idx, pred_idx) in enumerate(kf.split(Xc), start=1):
        # propensity model on the four other folds, predictions for this fold
        ps_model = learner().fit(Xc[fit_idx], W[fit_idx])
        e_hat[pred_idx] = ps_model.predict_proba(Xc[pred_idx])[:, 1]

        # outcome models, one per treatment arm
        treated = fit_idx[W[fit_idx] == 1]
        control = fit_idx[W[fit_idx] == 0]
        mu1_hat[pred_idx] = learner().fit(Xc[treated], Y[treated]).predict_proba(Xc[pred_idx])[:, 1]
        mu0_hat[pred_idx] = learner().fit(Xc[control], Y[control]).predict_proba(Xc[pred_idx])[:, 1]
        print(f"Fold {fold} of 5 done")

    print("\nOut-of-fold predictions ready for every customer.")
    print(f"Propensity model AUC (out of fold): {roc_auc_score(W, e_hat):.3f}")


## Overlap check

The histogram shows the estimated propensity scores for the two groups. Good overlap means that, at every value of the propensity score, both channels appear. If one group piles up at 0 or 1, those customers have no counterpart in the other group, and no estimator can compare them. The weights `1 / e(X)` and `1 / (1 - e(X))` then explode, so we keep only customers whose propensity lies between 0.05 and 0.95 (the common-support sample) and say clearly that the estimate applies to them only.


In [ ]:
if RUN_CAUSAL:
    plt.figure(figsize=(9, 5))
    bins = np.linspace(0, 1, 41)
    plt.hist(e_hat[W == 0], bins=bins, alpha=0.5, color='red', label='Telephone (control)')
    plt.hist(e_hat[W == 1], bins=bins, alpha=0.5, color='blue', label='Cellular (treated)')
    plt.xlabel('Estimated propensity score P(cellular | X), out of fold')
    plt.ylabel('Customers')
    plt.title('Overlap check')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Mean propensity, treated: {e_hat[W == 1].mean():.3f}; control: {e_hat[W == 0].mean():.3f}")
    common = (e_hat > 0.05) & (e_hat < 0.95)
    print(f"Customers with propensity <= 0.05: {(e_hat <= 0.05).sum()}; >= 0.95: {(e_hat >= 0.95).sum()}")
    print(f"Kept in the common-support sample: {common.sum()} of {n} ({common.mean():.1%})")

    # Where does the lack of overlap come from? Channel use by period (nr_employed identifies the quarter)
    print("\nCalls by channel and period (rows: nr_employed, a quarterly figure):")
    print(pd.crosstab(causal_data['nr_employed'], W, rownames=['nr_employed'], colnames=['cellular']))


In [ ]:
if RUN_CAUSAL:
    def aipw(Y, W, e, mu1, mu0):
        """Doubly robust score for each customer; returns the ATE and its standard error."""
        score = (mu1 - mu0
                 + W * (Y - mu1) / e
                 - (1 - W) * (Y - mu0) / (1 - e))
        return score.mean(), score.std(ddof=1) / np.sqrt(len(score))

    ate_cs, se_cs = aipw(Y[common], W[common], e_hat[common], mu1_hat[common], mu0_hat[common])
    naive_cs = Y[common & (W == 1)].mean() - Y[common & (W == 0)].mean()
    reg_only = (mu1_hat[common] - mu0_hat[common]).mean()

    print("Effect of a mobile-phone call versus a landline call on subscription, common-support sample:")
    print(f"  Naive difference in means:      {naive_cs:+.3f}")
    print(f"  Regression adjustment only:     {reg_only:+.3f}")
    print(f"  AIPW (doubly robust) estimate:  {ate_cs:+.3f}  (SE {se_cs:.3f}, "
          f"95% CI [{ate_cs - 1.96 * se_cs:+.3f}, {ate_cs + 1.96 * se_cs:+.3f}])")
    print(f"\nFor comparison, the naive difference on all customers was {naive_ate:+.3f}.")


## What did we learn?

In this run:

- The naive comparison on all customers gives +9.5 percentage points for mobile-phone calls (14.7 versus 5.2 percent subscribing).
- The propensity model predicts the channel very well (out-of-fold AUC 0.94), and the table above shows why: in one period (`nr_employed` = 5191.0, about 7,800 calls) the bank used landlines only, and in others almost only mobile phones. Overlap fails there. Only 57 percent of the customers have a propensity between 0.05 and 0.95.
- On that common-support sample the naive difference is +5.4 points and the AIPW estimate is +7.2 points (95% CI 6.0 to 8.4). Adjustment moved the number, so even within the overlapping periods the two channels were not used on the same kind of customers.

Questions to ask before believing +7 points:

- Is a channel effect plausible at all? Perhaps: a mobile call reaches the customer in person, a landline call may reach the household. But which customers gave the bank a mobile number? Customers with a university degree are in the cellular group about seven times in ten, customers with basic education a little over five times in ten. We adjust for age and education, not for income, occupation or how comfortable the customer is with the bank's digital services. Anything of that kind that also affects subscription is a confounder we cannot remove, and the unconfoundedness assumption fails.
- The estimate is for the 57 percent of customers in periods when both channels were used. What is the effect for the other 43 percent? The data cannot say.
- The macro indicators identify the period, and the period also carries the interest-rate environment and the campaign's history. Adjusting for the period is necessary, but it means we compare channels within a quarter, and a quarter with only a few hundred landline calls carries a lot of weight in the estimate. Does the confidence interval reflect that? Only if the models are right.

What would identify this properly: a randomised assignment of channel (an A/B test). If the bank assigned mobile or landline at random among customers who have both numbers, the naive difference in means would be the causal effect, with no modelling assumptions, and the machinery in this section would only be needed to sharpen the estimate.


### Recommended Reading

- **Book**: "Causal Inference: The Mixtape" by Scott Cunningham (free online), chapters on potential outcomes and matching.
- **Paper**: Athey and Imbens (2019), "Machine Learning Methods That Economists Should Know About", *Annual Review of Economics*.
